---
title: Linked-Read Metrics
subtitle: Linked-read metrics relating to barcodes and molecules
date: "9999-12-31"
edit_url: null
---

**Linked-read type**: PLACEHOLDER

In [ ]:
import altair as alt
from harpy.report.components import StatsBox, print_html, standard_itable
from harpy.report.utilities import binned_histogram_polars, nxx_polars
from harpy.report.theme import palette
import os
from pathlib import Path
import polars as pl
import sys
pl.Config.set_tbl_hide_column_data_types()



In [ ]:
indir = "reports/data/bxstats/"
platform = "haplotagging"


In [ ]:
def process_input(infile: str) -> dict:
    samplename = os.path.basename(infile).replace(".bxstats.gz", "")
    tb = pl.read_csv(infile, separator='\t').drop(["start", "end"])

    is_valid = pl.col('molecule') != "invalid"
    is_multiplex = is_valid & (pl.col('reads') >= 2)

    multiplex_df = tb.filter(is_multiplex)
    valid_df = tb.filter(is_valid)

    tot_mol = valid_df.height
    tot_valid_reads = valid_df['reads'].sum()
    tot_invalid_reads = tb.filter(~is_valid)['reads'].sum()
    tot_reads = tot_valid_reads + tot_invalid_reads
    if tot_reads == 0:
        return {
            "Sample": samplename,
            "Total Reads": 0,
            "Valid":  0,
            "Linked": 0,
            "Barcodes": 0,
            "Unique Molecules": 0,
            "Barcodes/Molecule": 0,
            "Reads/Molecule": 0,
            "Insert Molecule Coverage": 0,
            "BP Molecule Coverage": 0,
            "N50": 0,
            "N75": 0,
            "N90": 0
        }

    avg_reads_per_mol = round(multiplex_df['reads'].mean(), 1)
    avg_mol_cov = round(multiplex_df['coverage_inserts'].mean(), 2)
    avg_mol_cov_bp = round(multiplex_df['coverage_bp'].mean(), 2)

    _lens = multiplex_df['length_inferred'].sort(descending=True)

    n50 = nxx_polars(_lens, 50)
    n75 = nxx_polars(_lens, 75)
    n90 = nxx_polars(_lens, 90)

    tot_uniq_bx = tb['molecule'].str.replace(r'-\d+$', '', literal=False).n_unique()

    return {
        "Sample": samplename,
        "Total Reads": tot_reads,
        "Valid": round(tot_valid_reads / tot_reads, 4),
        "Linked": round(multiplex_df.height / tot_mol, 4),
        "Barcodes": tot_uniq_bx,
        "Unique Molecules": tot_mol,
        "Barcodes/Molecule": round(tot_mol / tot_uniq_bx, 2),
        "Reads/Molecule": avg_reads_per_mol,
        "Insert Molecule Coverage": avg_mol_cov,
        "BP Molecule Coverage": avg_mol_cov_bp,
        "N50": n50,
        "N75": n75,
        "N90": n90
    }


In [ ]:

files = [str(i) for i in Path(indir).glob("*.bxstats.gz")]
records = [process_input(i) for i in files]
aggregate_df = (
    pl.DataFrame(records, schema={
        "Sample": pl.String,
        "Total Reads": pl.Int64,
        "Valid": pl.Float64,
        "Linked": pl.Float64,
        "Barcodes": pl.Int64,
        "Unique Molecules": pl.Int64,
        "Barcodes/Molecule": pl.Float64,
        "Reads/Molecule": pl.Float64,
        "Insert Molecule Coverage": pl.Float64,
        "BP Molecule Coverage": pl.Float64,
        "N50": pl.Int64,
        "N75": pl.Int64,
        "N90": pl.Int64
    })
    .fill_nan(0)
    .fill_null(0)
    .sort("Sample")
)

This report aggregates the barcode-specific information from the alignments
that were created using `harpy align`. Detailed information for any one sample
can be found in that sample's individual report. The summary information spotlighted here represents averages across the samples.

In [ ]:
if len(aggregate_df) == 0:
    print_html("All input files were empty, no report to show")
    sys.exit(0)


In [ ]:
_n50 = round((aggregate_df['N50'].replace(0, None).mean() or 0) / 1000)
_n75 = round((aggregate_df['N75'].replace(0, None).mean() or 0) / 1000)
_n90 = round((aggregate_df['N90'].replace(0, None).mean() or 0) / 1000)
_bpm = round(aggregate_df['Barcodes/Molecule'].replace(0, None).mean() or 0, 2)
_bpmsd = round(aggregate_df['Barcodes/Molecule'].replace(0, None).std() or 0, 2)
_linked = round(aggregate_df['Linked'].mean() or 0, 3)
_linkedsd = round(aggregate_df['Linked'].std() or 0, 2)
_rpm = round(aggregate_df['Reads/Molecule'].replace(0, None).mean() or 0, 2)
_rpmsd = round(aggregate_df['Reads/Molecule'].replace(0, None).std() or 0, 2)

(
    StatsBox()
    .add(len(aggregate_df), "Samples")
    .add(_bpm, "Barcodes/Mol", plus_minus=_bpmsd)
    .conditional(_rpm, "Reads/Mol", 2, plus_minus=_rpmsd)
    .conditional(_linked, "Avg Linked", 0.4, as_percent=True)
    .conditional(aggregate_df['Valid'].replace(0, None).mean(), "Valid BX", 0.4, as_percent=True)
    .add(_n50, "N50 (kb)")
    .add(_n75, "N75 (kb)")
    .add(_n90, "N90 (kb)")
).render()

## Sample Information

The chart below is visual representation of some of the common linked-read  metrics that may help assess the success of sample/library preparation. For more details, see the table beneath the visualization and for even more granularity, you can inspect the report generated for each individual sample. 

In [ ]:
hover = alt.selection_point(
    fields=['Sample'],
    on='pointerover',  
    nearest=False,   
    empty=False
)

stroke_color = (
    alt.when(hover).then(alt.value("magenta"))
    .otherwise(alt.value(None))
)

(
    alt.Chart(aggregate_df[['Sample', 'Valid', 'Linked', 'Insert Molecule Coverage', 'BP Molecule Coverage']])
    .transform_fold(['Valid', 'Linked', 'Insert Molecule Coverage', 'BP Molecule Coverage'])
    .transform_calculate(random="random()")
    .mark_point(size=65)
    .encode(
        x=alt.X('value:Q', title='Percent').scale(domain=[0, 1]).axis(format="%"),
        y=alt.Y('random:Q', title=None).axis(labels=False, ticks=False, grid=False).scale(domain=[-0.1, 1.1]),
        color='key:N',
        shape='key:N',
        stroke=stroke_color,
        tooltip=[
            'Sample:N',
            alt.Tooltip('key:N', title="Metric"),
            alt.Tooltip('value:Q', title="Percent", format='.2%')
        ]
    )
    .add_params(hover)
    .properties(width=620, height=65)
    .facet(
        row=alt.Row('key:N', title=None, header=alt.Header(labelOrient='left', labelAngle = 0, labelAlign='left'))
    )
    .resolve_scale(y='independent')
    .properties(
        title=alt.Title('Per-Sample Metrics'),
        usermeta={'embedOptions': {'downloadFileName': 'alignments.per.sample'}}
    )
    .configure_legend(orient="top")
    .configure_view(stroke="#888888", strokeWidth=0.7)
)


This table [^1] is an aggregation of data for each sample based on their `*.bxstats.gz` file.
Every column after `Barcodes` ignores singletons in its calculations.

[^1]:
    | Column | Description |
    |:-------|:------------|
    | `Sample` | name of the sample |
    | `Total Reads`| total number of alignments |
    | `Valid` | proportion of valid barcoded alignments |
    | `Unique Molecules` | the unique DNA molecules as inferred from linked-read barcodes |
    | `Linked` | molecules composed of two or more single/paired-end sequences, in other words, molecules with linked-read information |
    | `Barcodes` | number of unique barcodes, which may differ from unique molecules after deconvolution |
    | `Barcodes/Molecule` | molecule-to-barcode ratio, which helps benchmark deconvolution performance, if performed |
    | `Reads/Molecule` | average number of reads per unique molecule |
    | `Insert Molecule Coverage` | average percent of a molecule that is covered by a read, where coverage includes unsequenced gaps between linked reads |
    | `BP Molecule Coverage` | average percent molecule coverage, where coverage only includes sequences and not the gaps between linked reads |
    | `N50` | N50 of inferred molecules |
    | `N75` | N75 of inferred molecules |
    | `N90` | N90 of inferred molecules |

In [ ]:
store = aggregate_df['Sample']
aggregate_df = aggregate_df.with_columns(
    pl.Series("Sample", [f'<a href="./{i}">{i}</a>' for i in aggregate_df['Sample']])
)
standard_itable(aggregate_df, "alignment.bx", fixedcols=2, html = True)


In [ ]:
aggregate_df = aggregate_df.with_columns(store)

## Library Performance
### NX metrics

The **NX** metrics (e.g. **N50**) tend to be useful for understanding centrality for molecule length in genomic contexts.
With your linked-read sequences aligned to a reference genome, you can make inferences about the lengths of the original molecules from which linked-read fragments were derived. These are the distributions of three common NX metrics (N50, N75, N90) across all samples are given below.

In [ ]:
selection = alt.selection_point(fields=['Nxx'], bind='legend')

(
    alt.Chart(aggregate_df[['Sample', 'N50', 'N75', 'N90']])
    .transform_fold(
        ['N50', 'N75', 'N90'],
        as_=['Nxx', 'value']
    )
    .transform_density(
        'value',
        groupby=['Nxx'],
        as_=['value', 'density']
    )
    .mark_area(interpolate = "basis")
    .encode(
        x=alt.X('value:Q', bin = "binned").axis(title='Length (kb)', labelExpr='datum.value / 1000'),
        y=alt.Y('density:Q', title='Density').axis(labels = False),
        color=alt.Color('Nxx:N', title='Metric').scale(domain = ['N50','N75','N90'])
    )
    .transform_filter(selection)
    .add_params(selection)
    .configure_legend(orient = "top")
    .properties(
        title=alt.Title('Nxx Values', subtitle = 'Values derived using non-singleton molecules'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.nxx'}}
    )
)


### Reads per molecule
This distribution shows the average number of reads per inferred molecule across your samples.

In [ ]:
_hist = binned_histogram_polars(aggregate_df['Reads/Molecule'], 0.25, True)
(
    alt.Chart(_hist)
    .mark_area(interpolate = "monotone")
    .encode(
        x=alt.X('interval:O', bin = 'binned').axis(title='Reads Per Molecule', tickMinStep=0.5, labelAngle=-40),
        y=alt.Y('proportion:Q', title='Percent of Samples').axis(format = '%'),
        tooltip = [
            alt.Tooltip('interval', title = "Reads Per Molecule", bin = "binned"),
            alt.Tooltip('proportion', title = "Percent of Samples").format('.1%')
        ]
    )
    .properties(
        title=alt.Title('Reads Per Molecule', subtitle = 'Values derived using non-singleton molecules'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.readspermol'}}
    )
)


### Valid barcodes
This is a distribution of what percent of total alignments each sample had valid barcodes[^3]:

[^3]:
    Valid barcodes with respect to the conventions of a given linked-read chemistry:
    
    **haplotagging**: `AxxCxxBxxDxx` where `xx` is not `00`
    
    **stLFR**: `X_Y_Z` where `X`, `Y`, or `Z` is not `0`
    
    **TELLseq**: there is no `N` nucleotide


In [ ]:
_hist = binned_histogram_polars(aggregate_df['Valid'], 0.1, True, 1)
(
    alt.Chart(_hist)
    .mark_area(interpolate = "monotone")
    .transform_filter(alt.datum.bin <= 1)
    .encode(
        x=alt.X('interval:O').axis(title='Proportion Valid Barcodes', labelAngle=-40),
        y=alt.Y('proportion:Q', title='Percent of Samples').axis(format = '%').scale(domain=[0,1]),
        tooltip = [
            alt.Tooltip('interval', title = "Proportion Valid Barcodes", bin = "binned"),
            alt.Tooltip('proportion', title = "Percent of Samples").format('.1%')
        ]
    )
    .properties(
            title=alt.Title('Proportion Valid Barcodes'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.percent.valid'}}
    )
)


### Total reads vs
This scatterplot is a diagnostic that shows the relationship between the proportion of total reads compared to _something_. Each point is a sample and is colored by the value of the variable you select.

In [ ]:
dropdownx = alt.binding_select(
    options=['Total Reads', "Valid", "Unique Molecules", "Linked", "Barcodes", "Barcodes/Molecule", "Reads/Molecule"],
    name='X: '
)
xcol_param = alt.param(
    value='Linked',
    bind=dropdownx
)

(
    alt.Chart(aggregate_df[['Sample', 'Valid', 'Total Reads', "Unique Molecules", "Linked", "Barcodes", "Barcodes/Molecule", "Reads/Molecule"]])
    .mark_circle()
    .add_params(xcol_param)
    .transform_calculate(Selected=f'datum[{xcol_param.name}]')
    .encode(
        x=alt.X('Selected:Q', title = ''),
        y='Total Reads:Q',
        color=alt.Color('Selected:Q').scale(scheme='viridis'),
        tooltip = ['Sample', 'Selected:Q', 'Total Reads:Q']
    )
    .properties(
        title=alt.Title('Proportion against Total Reads'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.percent.linked'}}
    )
)

## Supporting Info

:::{dropdown} NX values
`NX` is the length of the shortest molecule that when you sum the lengths of every molecule larger than it, represents at least **X%** of the total molecules by length. For example, `N50` would be the molecule length at which the sum of all the molecule lengths larger than it would
represent **50%** of the total molecules by length (sort of like a cumulative median).

As an example, imagine we had molecules with lengths [4, 3, 2, 1], making up a total length of 10. The `N50`
would be the first number (starting from the biggest) that sums up to at least 5 (50% of 10), which is `3`, because `4` + `3` = 7. The `N90` would be the first number that sums up to at least 9 (90% of 10), which is `2` because `4` + `3` + `2` = 9.
:::

:::{dropdown} Understanding inferred molecule lengths
Harpy uses the highest and lowest mapping positions of a read cluster sharing the same barcode (incorporating distance-based deconvolution thresholds, if configured to do so) to infer the original molecule length. Given that it's mostly impossible to understand how much of a source molecule the mapped reads actually covered, it would be more appropriate to think of the inferred molecule lengths (and NX measures) to be more like _"at least this long"_, since the reads likely did not originate from the ends of the source DNA molecule. The absolute length of the source DNA molecule would be a more useful diagnostic in troubleshooting/modifying the actual linked-read chemistry, whereas the the inferred molecule length describes what length of it is actually represented in the sequences and thus more useful in downstream/analytical contexts.
:::